<a href="https://colab.research.google.com/github/byuruth/Econ484/blob/main/Kaggle_competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Import necessary libraries
!git clone https://github.com/byuruth/Econ484.git
%cd Econ484/utilities
from preamble import *
%cd ../data

Load and Explore the Datasets

In [ ]:
train = pd.read_csv('train.csv')
train['SalePrice'].describe()

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
test = pd.read_csv('test.csv')
test.head()

Outcome to predict: Sale Price of home





Clean the data, quantify categorical variables

In [ ]:
import pandas as pd
import numpy as np

#low to high quality ranking
quality_map = {
    'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5
}

quality_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual', 'FireplaceQu',
    'GarageQual', 'GarageCond', 'PoolQC'
]

#additional variables with rankings
ordinal_maps = {
    'LotShape': {'IR3': 1, 'IR2': 2, 'IR1': 3, 'Reg': 4},
    'Utilities': {'ELO': 1, 'NoSeWa': 2, 'NoSewr': 3, 'AllPub': 4},
    'LandSlope': {'Sev': 1, 'Mod': 2, 'Gtl': 3},
    'BsmtExposure': {'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {
        'Unf': 1, 'LwQ': 2, 'Rec': 3,
        'BLQ': 4, 'ALQ': 5, 'GLQ': 6
    },
    'BsmtFinType2': {
        'Unf': 1, 'LwQ': 2, 'Rec': 3,
        'BLQ': 4, 'ALQ': 5, 'GLQ': 6
    },
    'GarageFinish': {'Unf': 1, 'RFn': 2, 'Fin': 3},
    'Functional': {
        'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4,
        'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8
    },
    'PavedDrive': {'N': 1, 'P': 2, 'Y': 3},
    'Fence': {'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4},
    'CentralAir': {'N': 0, 'Y': 1}
}

for col in quality_cols:
    ordinal_maps[col] = quality_map

#impute missing values with 0
none_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual',
    'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
    'MasVnrType'
]

for df in [train, test]:
    for col in none_cols:
        df[col] = df[col].fillna('None')

    for col, mapping in ordinal_maps.items():
        df[col] = df[col].replace({'None': 0}).map(
            {**mapping, 'None': 0}
        )

#treat as category
for df in [train, test]:
    df['MSSubClass'] = df['MSSubClass'].astype(str)

#impute again
numeric_cols = train.select_dtypes(
    include=np.number
).columns.drop('SalePrice')

for col in numeric_cols:
    median = train[col].median()
    train[col] = train[col].fillna(median)
    test[col] = test[col].fillna(median)

categorical_cols = train.select_dtypes(
    include=['object', 'category', 'string']
).columns

for col in categorical_cols:
    train[col] = train[col].fillna('Missing')
    test[col] = test[col].fillna('Missing')

#encode categorical variables
train = pd.get_dummies(
    train, columns=categorical_cols, dtype=int
)

test = pd.get_dummies(
    test, columns=categorical_cols, dtype=int
)

#realign
test = test.reindex(
    columns=train.drop(columns=['SalePrice']).columns,
    fill_value=0
)

test.head()


## Penalized Regression Prediction

In [ ]:
import pandas as pd
import numpy as np
from sklearn import linear_model
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

In [ ]:
#ols benchmark

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np

#prepare data
X = train.drop(columns=['SalePrice', 'Id'])
y = train['SalePrice']
scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

#cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Create OLS model
ols = LinearRegression().fit(X_scaled,y)
yhat = ols.predict(X_scaled)


In [ ]:
#Lasso

lassocv=linear_model.LassoCV(random_state=42).fit(X_scaled,y)
# print out the chosen value for alpha
print("Chosen alpha: {:.3f}".format(lassocv.alpha_))
# print the original number of regressors and the number selected by lasso
print("Number of regressors in the menu: ",len(X.columns))
print("Number of regressors selected by lasso: ",sum(lassocv.coef_!=0))
results = pd.DataFrame({'feature': X.columns[lassocv.coef_!=0],'coefficient': lassocv.coef_[lassocv.coef_!=0]})
results

In [ ]:
#Ridge
ridgecv=linear_model.RidgeCV(cv=5,alphas=(.1,1,10,50,100,1000)).fit(X_scaled,y)
print("Chosen alpha: {:.3f}".format(ridgecv.alpha_))
print("Accuracy on training set: {:.3f}".format(ridgecv.score(X_scaled,y)))
# look at the coefficients
results = pd.DataFrame({'feature': X.columns[ridgecv.coef_!=0],'coefficient': ridgecv.coef_[ridgecv.coef_!=0]})
results

In [ ]:
plt.plot(ridgecv.coef_, 's', label="RidgeCV")
plt.plot(lassocv.coef_, 'o', label="LassoCV")
plt.xlabel("Coefficient index")
plt.ylabel("Coefficient magnitude")
xlims = plt.xlim()
plt.hlines(0, xlims[0], xlims[1])
plt.xlim(xlims)
plt.legend()

In [ ]:
print("OLS RMSE: $", round(np.sqrt(ols_mse), 2))
print("Lasso RMSE: $", round(np.sqrt(lasso_mse), 2))
print("Ridge RMSE: $", round(np.sqrt(ridge_mse), 2))

it looks like OLS has the best in sample predictions (not really surprised by this). We can't see which one makes more accurate predictions out of sample, the the visual shoes us that lasso shrunk some of the coefficients all the way to zero.

## Random Forest Prediction

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = train.drop(columns=['SalePrice', 'Id'])
y = train['SalePrice']

#random forest
forest = RandomForestRegressor(n_estimators=5, random_state=2, bootstrap=True)
forest.fit(X, y)

#cross-validation
forest_cv_scores = cross_val_score(forest, X, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
forest_cv_mse = -forest_cv_scores.mean()

print("Random Forest Cross-Validated MSE:", forest_cv_mse)
print("Random Forest Cross-Validated RMSE:", np.sqrt(forest_cv_mse))


## Gradient Boosted Tree

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# Step 1: Define the parameters we want to test
param_grid = {
    'max_depth': [1, 2, 3, 4],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200]
}

# Step 2: Create the gradient boosted tree model
gbrt = GradientBoostingClassifier(random_state=0)

# Step 3: Set up grid search with 5-fold cross validation
grid_gbrt = GridSearchCV(gbrt, param_grid, cv=5)

# Step 4: Fit the model using training data
grid_gbrt.fit(X, y)

# Step 5: Display results
print("Best parameters:", grid_gbrt.best_params_)
print("Best CV accuracy:", grid_gbrt.best_score_)

In [ ]:
#make predictions
X_test = test[X.columns]

#standardize data
X_test_scaled = scaler.transform(X_test)

ols_predictions = ols.predict(X_test_scaled)
lasso_predictions = lassocv.predict(X_test_scaled)
ridge_predictions = ridgecv.predict(X_test_scaled)

#random forest prediction
forest_predictions = forest.predict(X_test)

#compare predictions
predictions = pd.DataFrame({
    'OLS': ols_predictions,
    'Lasso': lasso_predictions,
    'Ridge': ridge_predictions,
    'Random Forest': forest_predictions
})

predictions.head(10)


